# LLM verification: demo (matches slides 15–16)

**Pipeline (deployment time, no gold answer)**
1. **Rank pool:** Hybrid order or SSC order (solve-free)
2. **Solve candidate:** SCIP gives status, objective, non-zero variables
3. **Verify:** Sonnet 5.5, one call per candidate, returns a verdict and a confidence score
4. **Accept?** accept if verdict is correct and score ≥ 0.7, then return the answer
5. **Next or stop:** skip same-answer candidates; if none is accepted, abstain / regenerate

**Candidate level**
- **Correct candidate:** objective matches the benchmark gold within 10⁻⁴ (offline only)
- **Verdict:** verifier's decision, correct / incorrect
- **Confidence score:** verifier's self-reported confidence in [0, 1], uncalibrated
- **Accept:** verdict = correct and confidence score ≥ τ = 0.7
- **Skip:** a candidate whose LP-relaxation objective equals (rel. tol. 10⁻³) that of an already-rejected candidate
- **TPR (True Positive Rate):** fraction of correct candidates accepted; **FPR (False Positive Rate):** fraction of incorrect candidates accepted; **Precision:** fraction of accepted candidates that are correct

**Problem level (% of problems)**
- **Right:** returned candidate is correct; **Wrong:** returned candidate is incorrect; **Abstain:** no candidate accepted within the check budget
- **Accuracy when answered:** Right / (Right + Wrong); **Calls:** average verifier calls per problem

The notebook uses the saved verifier outputs in `runs/<bench>_verify/`, so no API key is needed. The gold answer is used only to label outcomes.

In [1]:
import sys, json, textwrap
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
import sim_verify as SV
from ssc.prompts_verify import verifier_blocks

BENCH, PID, TAU, BUDGET = "optmath", "optmath_0056", 0.7, 10
p = {x["pid"]: x for x in SV.load([BENCH])}[PID]
load = lambda *a: json.load(open(ROOT.joinpath("runs", *a)))
pool = load(BENCH, "pools", f"{PID}.json")
sols = load(f"{BENCH}_verify", "solutions", f"{PID}.json")["solutions"]
V = load(f"{BENCH}_verify", "verdicts", f"{PID}.json")["verdicts"]
gold = load(BENCH, "eval", f"{PID}.json")["gold"]
accept = lambda i: bool(V[i]["accept"]) and (V[i]["p_correct"] or 0) >= TAU

print(textwrap.fill(pool["question"], 110)[:700], "...\n")
print(f"Gold (offline only): {gold}   |   SSC top-1 = candidate {p['c1']}, objective {sols[p['c1']]['objective']}")

You are responsible for managing the distribution of resources across a network of nine cities. Each city has
a specific supply or demand for resources, and your goal is to minimize the total transportation cost while
ensuring that all cities receive the resources they need. The transportation costs vary depending on the route
taken between cities, and there are capacity limits on how much can be transported between any two cities.
City 0 has a net demand of 63 units of resources. City 1 has a net demand of 2 units of resources. City 2 has
a net demand of 5 units of resources. City 3 has a net demand of 16 units of resources. City 4 has a net
supply of 13 units of resources. City 5 has a net ...

Gold (offline only): 714.0   |   SSC top-1 = candidate 3, objective 700.0


In [2]:
# Step 1-3: both rankings, and the verifier's verdict / confidence score for each candidate
def show(order, name):
    print(f"--- {name}")
    print(f"{'rank':>4} {'cand':>4} {'LP obj':>8} {'MILP obj':>9} {'verdict':>10} {'score':>6} {'accept':>7}  correct (offline)")
    for r, i in enumerate(order, 1):
        o = sols[i]["objective"]
        print(f"{r:>4} {i:>4} {round(p['z'][i], 2) if p['z'][i] is not None else '-':>8} {round(o, 2) if o is not None else sols[i]['status']:>9} "
              f"{str(V[i]['verdict']):>10} {V[i]['p_correct'] or 0:6.2f} {str(accept(i)):>7}  {p['lab']['orig'][i]}")
show(SV.o_hybrid(p, 0.1), "Hybrid order")
show(SV.o_ssc(p), "SSC order")

--- Hybrid order
rank cand   LP obj  MILP obj    verdict  score  accept  correct (offline)
   1    3    700.0     700.0  incorrect   0.30   False  False
   2    6    714.0     714.0    correct   0.96    True  True
   3    9    700.0     700.0  incorrect   0.25   False  False
   4    5    700.0     700.0  incorrect   0.35   False  False
   5    2    700.0     700.0  incorrect   0.25   False  False
   6    4    700.0     700.0  incorrect   0.04   False  False
   7    8    714.0     714.0    correct   0.97    True  True
   8    0    700.0     700.0  incorrect   0.25   False  False
   9    1    700.0     700.0  incorrect   0.35   False  False
  10    7    700.0     700.0    correct   0.50   False  False
--- SSC order
rank cand   LP obj  MILP obj    verdict  score  accept  correct (offline)
   1    5    700.0     700.0  incorrect   0.35   False  False
   2    3    700.0     700.0  incorrect   0.30   False  False
   3    4    700.0     700.0  incorrect   0.04   False  False
   4    0    700.

In [3]:
# What the verifier sees and returns for the SSC top-1 candidate
c1 = p["c1"]
shown = verifier_blocks(pool["question"], pool["candidates"][c1]["code"], sols[c1])[1]["text"]
print("=== Solver result shown to the verifier ===")
print(shown.split("## Solver result for this model")[1].split("Work through")[0].strip())
print("\n=== Verifier output ===")
print("verdict:", V[c1]["verdict"], "| confidence score:", V[c1]["p_correct"], "| accept:", accept(c1))
for e in V[c1]["errors"]:
    print("error:", textwrap.fill(e, 105))

=== Solver result shown to the verifier ===
status: optimal
objective value: 700.0
non-zero decision variables:
x[0,4] = 6
x[0,5] = 13
x[0,6] = 10
x[0,7] = 19
x[0,8] = 15
x[1,4] = 2
x[2,4] = 5
x[3,6] = 16

=== Verifier output ===
verdict: incorrect | confidence score: 0.3 | accept: False
error: The flow-balance sign is inverted relative to the stated roles. The model sets outflow - inflow = +63 at
City 0, which makes the net-demand city a net sender. Supply cities 4-8 become net receivers (for example
x[0,4], x[0,5], x[0,8]). A demand city should have inflow - outflow = demand, and a supply city should
have outflow - inflow = supply. The solution shows City 0 sending 63 units out, which contradicts the
text. The arc costs are asymmetric, so the correct orientation (flows into City 0 from 4-8) would very
likely give a different optimal cost than 700.


In [4]:
# Steps 4-5: walk both policies (skip same-answer candidates, stop at the first accept, budget 10)
def walk(order, name):
    calls, rejected = 0, []
    print(f"--- {name}")
    for i in order:
        if calls >= BUDGET:
            break
        if any(SV.same(p["z"][i], p["z"][j]) for j in rejected):
            print(f"  skip  cand {i}  (same answer as a rejected candidate, no call)")
            continue
        calls += 1
        print(f"  check {calls}: cand {i}  obj={sols[i]['objective']}  verdict={V[i]['verdict']}  score={V[i]['p_correct']}  -> {'ACCEPT' if accept(i) else 'reject'}")
        if accept(i):
            print(f"  => return {sols[i]['objective']}  | correct (offline): {p['lab']['orig'][i]}  | calls: {calls}\n"); return
        rejected.append(i)
    print(f"  => abstain  | calls: {calls}\n")

walk(SV.o_hybrid(p, 0.1), "Hybrid + skip")
walk(SV.o_ssc(p), "SSC order + skip")

--- Hybrid + skip
  check 1: cand 3  obj=700.0  verdict=incorrect  score=0.3  -> reject
  check 2: cand 6  obj=714.0  verdict=correct  score=0.96  -> ACCEPT
  => return 714.0  | correct (offline): True  | calls: 2

--- SSC order + skip
  check 1: cand 5  obj=700.0  verdict=incorrect  score=0.35  -> reject
  skip  cand 3  (same answer as a rejected candidate, no call)
  skip  cand 4  (same answer as a rejected candidate, no call)
  skip  cand 0  (same answer as a rejected candidate, no call)
  skip  cand 1  (same answer as a rejected candidate, no call)
  skip  cand 7  (same answer as a rejected candidate, no call)
  skip  cand 2  (same answer as a rejected candidate, no call)
  skip  cand 9  (same answer as a rejected candidate, no call)
  check 2: cand 6  obj=714.0  verdict=correct  score=0.96  -> ACCEPT
  => return 714.0  | correct (offline): True  | calls: 2



## Results: MaMO-Complex + OptMath, 200 problems × 10 candidates, τ = 0.7

In [5]:
from ssc.eval_top2 import load_benchmark, pick_hybrid
BENCHES = ["mamo_complex", "optmath"]
P = SV.load(BENCHES); N = len(P)
raw = {x["pid"]: x for b in BENCHES for x in load_benchmark(str(ROOT), b, "scores_v1")}
acc_ = lambda q, i: bool(q["acc"][i]) and q["pc"][i] >= TAU

# 1) verifier quality (candidate level)
A = [(acc_(q, i), y) for q in P for i, y in enumerate(q["lab"]["orig"])]
tp = sum(a and y for a, y in A); fp = sum(a and not y for a, y in A)
npos = sum(y for _, y in A); nneg = len(A) - npos
nocov = [q for q in P if not any(q["lab"]["orig"])]
fa = sum(any(acc_(q, i) for i in range(len(q["acc"]))) for q in nocov)
print("Verifier   TPR    FPR    Precision   No-correct pools with a false accept")
print(f"Sonnet 5.5 {tp/npos:.2f}   {fp/nneg:.2f}   {tp/(tp+fp):.2f}        {fa} / {len(nocov)}\n")

# 2) verification policies (problem level)
def policy(order_fn):
    calls = right = wrong = 0
    for q in P:
        c, o = SV.run_policy(q, order_fn(q), q["lab"]["orig"], TAU, BUDGET, True)
        calls += c; right += o == "right"; wrong += o == "wrong"
    return calls / N, 100 * right / N, 100 * wrong / N, 100 * (N - right - wrong) / N, 100 * right / (right + wrong)
pol = {"Hybrid + skip": policy(lambda q: SV.o_hybrid(q, 0.1)), "SSC order + skip": policy(SV.o_ssc)}
print(f"{'Policy':18s} {'Calls':>6} {'Right':>6} {'Wrong':>6} {'Abstain':>8} {'Acc. when answered':>19}")
for k, (c, r, w, a, acc) in pol.items():
    print(f"{k:18s} {c:6.2f} {r:6.1f} {w:6.1f} {a:8.1f} {acc:19.1f}")

# 3) comparison with previous methods
acc1 = 100 * sum(q["lab"]["orig"][q["c1"]] for q in P) / N
acc2 = 100 * sum(q["lab"]["orig"][q["c1"]] or q["lab"]["orig"][pick_hybrid(raw[q["pid"]], q["c1"], lam=0.1, delta=1e-3)] for q in P) / N
cov = 100 * sum(any(q["lab"]["orig"]) for q in P) / N
print(f"\n{'Method':44s} {'Answers':>13} {'Calls':>6} {'Right':>6} {'Wrong':>6} {'Abstain':>8}")
print(f"{'SSC (Acc@1)':44s} {'1':>13} {0:6} {acc1:6.1f} {100-acc1:6.1f} {0:8.1f}")
print(f"{'Hybrid (Acc@2, right if either is correct)':44s} {'2':>13} {0:6} {acc2:6.1f} {100-acc2:6.1f} {0:8.1f}")
for k, (c, r, w, a, _) in pol.items():
    print(f"{k:44s} {'1 or abstain':>13} {c:6.2f} {r:6.1f} {w:6.1f} {a:8.1f}")
print(f"{'Pool coverage (upper bound)':44s} {'-':>13} {'-':>6} {cov:6.1f}")

Verifier   TPR    FPR    Precision   No-correct pools with a false accept
Sonnet 5.5 0.82   0.32   0.68        48 / 65

Policy              Calls  Right  Wrong  Abstain  Acc. when answered
Hybrid + skip        1.62   45.5   22.0     32.5                67.4
SSC order + skip     1.62   45.5   22.5     32.0                66.9

Method                                             Answers  Calls  Right  Wrong  Abstain
SSC (Acc@1)                                              1      0   51.0   49.0      0.0
Hybrid (Acc@2, right if either is correct)               2      0   60.0   40.0      0.0
Hybrid + skip                                 1 or abstain   1.62   45.5   22.0     32.5
SSC order + skip                              1 or abstain   1.62   45.5   22.5     32.0
Pool coverage (upper bound)                              -      -   67.5
